# 👁️ Day 01a: Observer & Strategy Patterns

---

## 🎯 What You'll Master Today
- **Observer**: one-to-many event notification (pub/sub)
- **Strategy**: swap algorithms at runtime without changing client code
- Real-world: event systems, notification services, payment processing, sorting

---

## 🎭 The Analogies

**Observer** = YouTube subscriptions  
You subscribe to a channel. When a new video drops, ALL subscribers get notified. The channel doesn't know or care who's subscribed.

**Strategy** = GPS navigation modes  
Same destination. You pick "fastest route", "shortest route", or "avoid tolls". The navigation app uses whichever algorithm you choose — the app itself doesn't change.

---

In [ ]:
# ============================================
# 1. OBSERVER — Event System
# ============================================
from abc import ABC, abstractmethod

class Observer(ABC):
    @abstractmethod
    def update(self, event: str, data: dict) -> None: pass

class EventEmitter:
    """Subject / Observable — manages subscribers."""
    def __init__(self):
        self._listeners: dict[str, list[Observer]] = {}
    
    def on(self, event: str, observer: Observer):
        self._listeners.setdefault(event, []).append(observer)
    
    def off(self, event: str, observer: Observer):
        if event in self._listeners:
            self._listeners[event].remove(observer)
    
    def emit(self, event: str, data: dict = None):
        for observer in self._listeners.get(event, []):
            observer.update(event, data or {})


# Concrete observers
class EmailAlert(Observer):
    def update(self, event: str, data: dict):
        print(f"  📧 Email: [{event}] {data}")

class SlackAlert(Observer):
    def update(self, event: str, data: dict):
        print(f"  💬 Slack: [{event}] {data}")

class DashboardUpdate(Observer):
    def update(self, event: str, data: dict):
        print(f"  📊 Dashboard: [{event}] {data}")


# Usage
store = EventEmitter()
email = EmailAlert()
slack = SlackAlert()
dash = DashboardUpdate()

store.on("order_placed", email)
store.on("order_placed", slack)
store.on("order_placed", dash)
store.on("payment_failed", email)
store.on("payment_failed", slack)

print("Order placed:")
store.emit("order_placed", {"order_id": 42, "total": 99.99})

print("\nPayment failed:")
store.emit("payment_failed", {"order_id": 42, "reason": "insufficient funds"})

In [ ]:
# ============================================
# 2. STRATEGY — Payment Processing
# ============================================
from abc import ABC, abstractmethod

# Strategy interface
class PaymentStrategy(ABC):
    @abstractmethod
    def pay(self, amount: float) -> str: pass

# Concrete strategies
class CreditCardPayment(PaymentStrategy):
    def __init__(self, card_number: str):
        self._card = card_number[-4:]  # last 4 digits
    def pay(self, amount: float) -> str:
        return f"💳 Paid ${amount:.2f} with card ending {self._card}"

class UPIPayment(PaymentStrategy):
    def __init__(self, upi_id: str):
        self._upi = upi_id
    def pay(self, amount: float) -> str:
        return f"📱 Paid ${amount:.2f} via UPI ({self._upi})"

class CryptoPayment(PaymentStrategy):
    def __init__(self, wallet: str):
        self._wallet = wallet[:8] + "..."
    def pay(self, amount: float) -> str:
        return f"🪙 Paid ${amount:.2f} from wallet {self._wallet}"


# Context — uses strategy without knowing which one
class ShoppingCart:
    def __init__(self):
        self._items: list[tuple[str, float]] = []
        self._strategy: PaymentStrategy = None
    
    def add_item(self, name: str, price: float):
        self._items.append((name, price))
    
    def set_payment(self, strategy: PaymentStrategy):
        self._strategy = strategy
    
    def checkout(self) -> str:
        total = sum(price for _, price in self._items)
        if not self._strategy:
            return "❌ No payment method selected"
        return self._strategy.pay(total)


cart = ShoppingCart()
cart.add_item("Laptop", 999.99)
cart.add_item("Mouse", 29.99)

# Swap strategy at runtime!
cart.set_payment(CreditCardPayment("4111111111111234"))
print(cart.checkout())

cart.set_payment(UPIPayment("sarthak@paytm"))
print(cart.checkout())

cart.set_payment(CryptoPayment("0xABC123DEF456"))
print(cart.checkout())

In [ ]:
# ============================================
# 3. Pythonic Strategy — Using Functions
# ============================================
from typing import Callable

# In Python, you don't always need classes!
def sort_by_price(items):
    return sorted(items, key=lambda x: x["price"])

def sort_by_rating(items):
    return sorted(items, key=lambda x: x["rating"], reverse=True)

def sort_by_name(items):
    return sorted(items, key=lambda x: x["name"])


def display_products(products: list[dict], strategy: Callable):
    """Context — accepts any sorting function."""
    for p in strategy(products):
        print(f"  {p['name']}: ${p['price']} ⭐{p['rating']}")


products = [
    {"name": "Phone", "price": 699, "rating": 4.5},
    {"name": "Tablet", "price": 399, "rating": 4.8},
    {"name": "Laptop", "price": 1299, "rating": 4.2},
]

print("By price:")
display_products(products, sort_by_price)

print("\nBy rating:")
display_products(products, sort_by_rating)

print("\nBy name:")
display_products(products, sort_by_name)

```
╔════════════════════════════════════════════════════════╗
║             PATTERN COMPARISON MAP                     ║
╠═══════════════╦════════════════════════════════════════╣
║  Aspect        ║  Observer vs Strategy                  ║
╠═══════════════╬════════════════════════════════════════╣
║  Relationship  ║  Observer: 1-to-many                   ║
║                ║  Strategy: 1-to-1                      ║
║  Direction     ║  Observer: subject → observers (push)  ║
║                ║  Strategy: context → strategy (call)   ║
║  When          ║  Observer: react to state changes      ║
║                ║  Strategy: pick algorithm at runtime   ║
║  Python way    ║  Observer: signals/events/callbacks    ║
║                ║  Strategy: pass functions/callables    ║
╚═══════════════╩════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Observer vs Pub/Sub? | Observer is in-process, direct references. Pub/Sub uses message broker, decoupled |
| 2 | How to avoid memory leaks in Observer? | Use weak references or explicit unsubscribe. Remove dead observers |
| 3 | Strategy vs if/elif chains? | Strategy is OCP-compliant — add new strategies without modifying existing code |
| 4 | Pythonic Strategy? | Pass functions as arguments. No need for classes when a callable suffices |
| 5 | Real-world Observer examples? | Django signals, React state management, event-driven microservices |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Observer = publish events, notify subscribers         │
## │  • Strategy = swap algorithms without changing context   │
## │  • Both follow Open/Closed Principle                     │
## │  • Python idiom: use functions as strategies             │
## │  • Observer enables loose coupling between components    │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Command & State** — encapsulate requests and manage state machines